# 🏆 Notebook 4 : Benchmark, Optimisation & Explicabilité des Modèles
**Projet :** Prédiction de l'Attrition des Employés — Data Mining  
**Objectif :** Évaluer et comparer 6 algorithmes de classification supervisée (Zero-R, KNN, Arbre de Décision, Forêt Aléatoire, Naive Bayes, SVM) via validation croisée stratifiée 5-Fold, analyse ROC/AUC, matrices de confusion et explicabilité SHAP.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score
from joblib import load

sns.set_theme(style="whitegrid")
print("Environnement d'évaluation prêt.")

## 1. Chargement des Résultats et Métriques du Pipeline

In [ ]:
RESULTS_DIR = Path("../results")
MODELS_DIR = Path("../models")

with open(RESULTS_DIR / "metrics.json", "r") as f:
    metrics = json.load(f)

with open(RESULTS_DIR / "cv_results_summary.json", "r") as f:
    cv_results = json.load(f)

with open(RESULTS_DIR / "best_hyperparameters.json", "r") as f:
    best_params = json.load(f)

# Tableau récapitulatif
df_metrics = pd.DataFrame(metrics).T
df_cv = pd.DataFrame(cv_results).T
df_benchmark = df_metrics.join(df_cv)
df_benchmark[["accuracy", "precision", "recall", "f1", "roc_auc", "mean_cv_f1", "std_cv_f1"]]

## 2. Courbes ROC Comparatives

In [ ]:
with open(RESULTS_DIR / "roc_data.json", "r") as f:
    roc_data = json.load(f)

plt.figure(figsize=(10, 7))
for name, d in roc_data.items():
    plt.plot(d["fpr"], d["tpr"], label=f"{name} (AUC = {d['auc']:.3f})", lw=2)

plt.plot([0, 1], [0, 1], "k--", label="Aléatoire (AUC = 0.500)")
plt.xlabel("Taux de Faux Positifs (FPR)")
plt.ylabel("Taux de Vrais Positifs (TPR / Recall)")
plt.title("Courbes ROC Comparatives — 6 Modèles de Classification")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 3. Matrices de Confusion sur le Jeu de Test

In [ ]:
with open(RESULTS_DIR / "confusion_matrices.json", "r") as f:
    cms = json.load(f)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, (name, cm) in enumerate(cms.items()):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[i],
                xticklabels=["Reste (0)", "Quitte (1)"], yticklabels=["Reste (0)", "Quitte (1)"])
    axes[i].set_title(f"Matrice : {name}")
    axes[i].set_xlabel("Prédiction")
    axes[i].set_ylabel("Réalité")

plt.tight_layout()
plt.show()

## 4. Importance des Variables (Feature Importance - Arbres & Forêt)

In [ ]:
with open(RESULTS_DIR / "feature_importances.json", "r") as f:
    fi = json.load(f)

rf_fi = pd.Series(fi["RandomForest"]).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
rf_fi.plot(kind="barh", color="#4F46E5")
plt.title("Importance des Caractéristiques — Random Forest (Impureté de Gini)")
plt.xlabel("Importance Relative")
plt.tight_layout()
plt.show()

## 5. Explicabilité Globale avec SHAP (SHAP Values)

In [ ]:
import shap

# Chargement du modèle Random Forest
rf_pipeline = load(MODELS_DIR / "RandomForest_pipeline.joblib")
preprocessor = rf_pipeline.named_steps["preprocessor"]
model = rf_pipeline.named_steps["classifier"]

test_data = pd.read_csv("../data/processed/test_set.csv")
X_test = test_data[rf_pipeline.feature_names_in_]
X_test_trans = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

# Calcul des explications SHAP
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_trans[:300])

# Summary plot
if isinstance(shap_values, list):
    # Classe positive (1: Attrition = Yes)
    sv = shap_values[1]
else:
    sv = shap_values[:, :, 1] if len(shap_values.shape) == 3 else shap_values

shap.summary_plot(sv, X_test_trans[:300], feature_names=feature_names, show=False)
plt.title("Impact des Variables sur l'Attrition RH (SHAP Summary Plot)", pad=20)
plt.tight_layout()
plt.show()

## 6. Synthèse des Résultats & Recommandations RH
1. **Performance Modèles :** L'Arbre de Décision optimisé et la Forêt Aléatoire obtiennent les meilleurs compromis F1 et ROC-AUC sur la classe minoritaire.
2. **Facteurs Prépondérants :** Le Revenu Mensuel (`MonthlyIncome`), la Distance Domicile-Travail (`DistanceFromHome`), l'Âge (`Age`) et les Heures Supplémentaires (`OverTime`) constituent les déterminants majeurs de l'attrition.
3. **Plan d'Action RH :** Revaloriser les rémunérations des profils clés, encadrer strictement les heures supplémentaires et proposer des solutions de flexibilité/télétravail pour les collaborateurs éloignés.